In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Making Benchmarking File

In [2]:
!pip install -q google-generativeai sentence-transformers faiss-gpu networkx pydantic scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 14.0 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install -q --upgrade chromadb opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 73.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [4]:
import json

benchmark_data = [
    # --- Original 5 Cases ---
    {
        "case_id": "ONC_001",
        "cancer_type": "lung cancer",
        "histology": "adenocarcinoma",
        "conversation": "Patient is a 62-year-old male with a 3.2 cm solitary left lower lobe mass. Biopsy confirmed adenocarcinoma. Liquid biopsy identified EGFR Exon 19 deletion. ECOG is 0. Moderate smoking history. No known distant metastases.",
        "ground_truth": {
            "primary_biomarker": "EGFR",
            "indicated_therapies": ["Osimertinib", "Erlotinib", "Gefitinib"],
            "mandatory_imaging": ["Brain MRI", "PET/CT", "Chest CT"],
            "mandatory_labs_workup": ["EBUS-TBNA", "Pulmonary Function Tests", "CMP", "CBC"]
        }
    },
    {
        "case_id": "ONC_002",
        "cancer_type": "breast cancer",
        "histology": "invasive ductal carcinoma",
        "conversation": "48-year-old premenopausal female presenting with a palpable 2.5 cm right breast lump and palpable ipsilateral axillary node. Core needle biopsy confirms invasive ductal carcinoma, ER negative, PR negative, and HER2 3+ on IHC. ECOG 0.",
        "ground_truth": {
            "primary_biomarker": "HER2",
            "indicated_therapies": ["Trastuzumab", "Pertuzumab", "T-DM1"],
            "mandatory_imaging": ["Breast MRI", "Mammogram", "PET/CT", "Echocardiogram"],
            "mandatory_labs_workup": ["Core Biopsy Sentinel Node", "CBC", "LFTs"]
        }
    },
    {
        "case_id": "ONC_003",
        "cancer_type": "colorectal cancer",
        "histology": "adenocarcinoma",
        "conversation": "59-year-old male presenting with intermittent hematochezia and bowel habit changes. Colonoscopy revealed an obstructing sigmoid colon lesion; pathology shows adenocarcinoma. CT demonstrates 2 synchronous hepatic lesions. KRAS codon 12 wild-type, BRAF V600E negative, MSS. ECOG 1.",
        "ground_truth": {
            "primary_biomarker": "KRAS",
            "indicated_therapies": ["Cetuximab", "Panitumumab", "FOLFOX", "FOLFIRI"],
            "mandatory_imaging": ["CT Chest/Abdomen/Pelvis", "Liver MRI", "Colonoscopy"],
            "mandatory_labs_workup": ["CEA", "CBC", "CMP", "MSI testing"]
        }
    },
    {
        "case_id": "ONC_004",
        "cancer_type": "prostate cancer",
        "histology": "adenocarcinoma",
        "conversation": "71-year-old male with urinary hesitancy. Serum PSA is 38 ng/mL. Transrectal ultrasound-guided biopsy demonstrates Gleason score 4+5=9 prostate adenocarcinoma involving 8 of 12 cores. ECOG 0. Bone pain in lumbar spine.",
        "ground_truth": {
            "primary_biomarker": "PSA",
            "indicated_therapies": ["Leuprolide", "Enzalutamide", "Abiraterone"],
            "mandatory_imaging": ["PSMA PET/CT", "Bone Scan", "Pelvic MRI"],
            "mandatory_labs_workup": ["Serum PSA", "Alkaline Phosphatase", "CBC", "CMP", "BRCA testing"]
        }
    },
    {
        "case_id": "ONC_005",
        "cancer_type": "melanoma",
        "histology": "cutaneous melanoma",
        "conversation": "54-year-old female presenting with an enlarging, pigmented lesion on the mid-back. Excisional biopsy shows nodular melanoma, Breslow depth 3.8 mm, with ulceration. Molecular testing reveals BRAF V600E mutation. ECOG 0.",
        "ground_truth": {
            "primary_biomarker": "BRAF",
            "indicated_therapies": ["Dabrafenib", "Trametinib", "Pembrolizumab", "Nivolumab"],
            "mandatory_imaging": ["Brain MRI", "PET/CT"],
            "mandatory_labs_workup": ["Sentinel Lymph Node Biopsy", "LDH", "Skin Examination"]
        }
    },
    
    # --- New 5 Cases ---
    {
        "case_id": "ONC_006",
        "cancer_type": "pancreatic cancer",
        "histology": "ductal adenocarcinoma",
        "conversation": "65-year-old male presents with painless jaundice and weight loss. CT abdomen reveals a 4 cm mass in the head of the pancreas wrapping the SMA. EUS-FNA confirms ductal adenocarcinoma. CA 19-9 is severely elevated at 1500 U/mL. Germline testing pending. ECOG 1.",
        "ground_truth": {
            "primary_biomarker": "CA 19-9",
            "indicated_therapies": ["FOLFIRINOX", "Gemcitabine", "Abraxane"],
            "mandatory_imaging": ["Pancreatic Protocol CT", "Chest CT", "PET/CT"],
            "mandatory_labs_workup": ["Serum CA 19-9", "Bilirubin/LFTs", "BRCA germline testing", "Diagnostic Laparoscopy"]
        }
    },
    {
        "case_id": "ONC_007",
        "cancer_type": "small cell lung cancer",
        "histology": "small cell carcinoma",
        "conversation": "70-year-old heavy smoker presents with superior vena cava syndrome and a large central mediastinal mass. Biopsy confirms small cell carcinoma. CT shows multiple hepatic lesions. ECOG 2.",
        "ground_truth": {
            "primary_biomarker": "None",
            "indicated_therapies": ["Etoposide", "Carboplatin", "Cisplatin", "Atezolizumab", "Durvalumab"],
            "mandatory_imaging": ["Brain MRI", "PET/CT", "Chest/Abdomen/Pelvis CT"],
            "mandatory_labs_workup": ["CBC", "CMP", "LDH"]
        }
    },
    {
        "case_id": "ONC_008",
        "cancer_type": "gastrointestinal stromal tumor",
        "histology": "spindle cell neoplasm",
        "conversation": "55-year-old female presents with anemia and melena. Endoscopy finds an ulcerated subepithelial gastric mass. Resection shows a 6 cm spindle cell neoplasm. IHC is strongly positive for CD117 (KIT). Mitotic rate is 8/50 HPF. Mutational analysis shows KIT exon 11 mutation. ECOG 0.",
        "ground_truth": {
            "primary_biomarker": "KIT",
            "indicated_therapies": ["Imatinib", "Sunitinib", "Regorafenib"],
            "mandatory_imaging": ["CT Abdomen/Pelvis", "Chest CT"],
            "mandatory_labs_workup": ["Endoscopy", "CBC", "CMP", "KIT mutational analysis"]
        }
    },
    {
        "case_id": "ONC_009",
        "cancer_type": "cholangiocarcinoma",
        "histology": "adenocarcinoma",
        "conversation": "60-year-old male evaluated for right upper quadrant pain. MRI liver demonstrates a 5 cm intrahepatic mass with peripheral enhancement. Core biopsy confirms adenocarcinoma consistent with cholangiocarcinoma. NGS panel reveals an FGFR2 fusion. ECOG 1.",
        "ground_truth": {
            "primary_biomarker": "FGFR2",
            "indicated_therapies": ["Pemigatinib", "Gemcitabine", "Cisplatin"],
            "mandatory_imaging": ["MRI Liver with IV Contrast", "CT Chest/Abdomen/Pelvis"],
            "mandatory_labs_workup": ["CA 19-9", "CEA", "Hepatitis Panel", "LFTs"]
        }
    },
    {
        "case_id": "ONC_010",
        "cancer_type": "ovarian cancer",
        "histology": "high-grade serous carcinoma",
        "conversation": "63-year-old female presents with abdominal bloating and early satiety. Ultrasound shows a complex 8 cm left adnexal mass with massive ascites. Paracentesis cytology is positive for adenocarcinoma. She has a known deleterious BRCA1 germline mutation. ECOG 1.",
        "ground_truth": {
            "primary_biomarker": "BRCA1",
            "indicated_therapies": ["Carboplatin", "Paclitaxel", "Olaparib", "Bevacizumab"],
            "mandatory_imaging": ["CT Chest/Abdomen/Pelvis", "Pelvic Ultrasound or MRI"],
            "mandatory_labs_workup": ["CA-125", "BRCA germline panel", "CBC", "CMP"]
        }
    }
]

# Write to Kaggle working directory
output_path = "/kaggle/working/oncology_bench.jsonl"
with open(output_path, "w") as f:
    for item in benchmark_data:
        f.write(json.dumps(item) + "\n")

print(f"Successfully created {output_path} with {len(benchmark_data)} reference benchmark cases.")

Successfully created /kaggle/working/oncology_bench.jsonl with 10 reference benchmark cases.


In [5]:
def calculate_concordance(generated_text: str, ground_truth_items: list[str]) -> float:
    """Calculates recall of mandatory medical workup items."""
    if not ground_truth_items:
        return 1.0
    text_lower = generated_text.lower()
    matches = sum(1 for item in ground_truth_items if item.lower() in text_lower)
    return matches / len(ground_truth_items)

def calculate_kg_citation_accuracy(top_paths: list[dict], generated_text: str) -> float:
    """Calculates whether retrieved KG entities are explicitly cited in the final synthesis."""
    if not top_paths:
        return 0.0
    text_lower = generated_text.lower()
    cited = 0
    for p in top_paths:
        target_node = str(p.get("node", "")).lower()
        if target_node and target_node in text_lower:
            cited += 1
    return cited / len(top_paths)

# Final Branch Code

In [6]:
# =============================================================================
# FINAL BRANCH: Automated 10-Patient Oncology RAG Benchmark & LLM-as-a-Judge
# =============================================================================

import os
import gc
import json
import time
import shutil
import numpy as np
import networkx as nx
import faiss
import torch
from typing import Optional, Literal
from pydantic import BaseModel, Field
from sentence_transformers import CrossEncoder, SentenceTransformer
import chromadb
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, GoogleAPICallError

# ==========================================
# 0. SETUP & API CONFIGURATION
# ==========================================
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY")
genai.configure(api_key=gemini_api_key)

# We use gemini-1.5-flash for high throughput and rate-limit headroom
model = genai.GenerativeModel('gemini-3.1-flash-lite')

DATA_DIR = "/kaggle/input/datasets/nilanjansaha123hello/oncology-kg-data"
CHROMA_SRC = "/kaggle/input/datasets/nilanjansaha123hello/chroma-db-oncology-oxford"
CHROMA_WORK = "/kaggle/working/chroma_db_bench"
BENCHMARK_PATH = "/kaggle/working/oncology_bench.jsonl"

def call_gemini_with_retry(prompt: str, generation_config=None, max_retries: int = 5, base_delay: float = 4.0):
    """Executes Gemini API calls with exponential backoff to handle free/standard tier rate limits."""
    delay = base_delay
    for attempt in range(max_retries):
        try:
            if generation_config:
                response = model.generate_content(prompt, generation_config=generation_config)
            else:
                response = model.generate_content(prompt)
            time.sleep(2.0)  # Safe inter-call cooldown
            return response.text
        except (ResourceExhausted, GoogleAPICallError) as e:
            print(f"Rate limit hit ({e}). Backing off for {delay:.1f}s (Attempt {attempt+1}/{max_retries})...")
            time.sleep(delay)
            delay *= 2.0
    raise RuntimeError("Exceeded maximum Gemini API retries due to rate limits.")


# ==========================================
# 1. ENTITY EXTRACTION SCHEMA
# ==========================================
class Biomarker(BaseModel):
    gene_or_marker: str
    variant: Optional[str] = None
    value: Optional[str] = None
    source: Literal["reported", "suspected"] = "suspected"

class OncologyEntities(BaseModel):
    cancer_type: Optional[str] = None
    primary_site: Optional[str] = None
    histology: Optional[str] = None
    stage_tnm: Optional[str] = None
    biomarkers: list[Biomarker] = Field(default_factory=list)
    ecog_status: Optional[int] = None
    prior_treatments: list[str] = Field(default_factory=list)
    comorbidities: list[str] = Field(default_factory=list)
    symptoms: list[str] = Field(default_factory=list)

def extract_oncology_entities(conversation: str) -> OncologyEntities:
    prompt = f"""You are extracting structured oncology entities from a patient-provider conversation.
Return ONLY a valid JSON object matching this schema (omit fields with no evidence):
{json.dumps(OncologyEntities.model_json_schema())}

Conversation:
{conversation}"""
    
    cfg = genai.GenerationConfig(response_mime_type="application/json", temperature=0.0)
    raw_json = call_gemini_with_retry(prompt, generation_config=cfg)
    return OncologyEntities.model_validate_json(raw_json)


# ==========================================
# 2. LOAD PERSISTENT MODELS & DATABASES (ONCE)
# ==========================================
print("=== Initializing Persistent Graph & Vector Resources ===")

# A. PrimeKG & Hybrid Retriever
print("1/4 Loading PrimeKG NetworkX Graph...")
prime_kg = nx.read_graphml(os.path.join(DATA_DIR, "primekg.graphml"))

RELATION_WEIGHTS = {
    "indication": 1.0, "off-label use": 1.0, "contraindication": 1.0,
    "disease_protein": 1.0, "drug_protein": 1.0, "disease_phenotype_positive": 0.8,
    "disease_disease": 0.5, "drug_effect": 0.5, "protein_protein": 0.2
}
HUB_DEGREE_THRESHOLD = 200

def type_aware_bfs(kg: nx.DiGraph, source: str, depth: int = 3) -> nx.DiGraph:
    visited, frontier = {source}, [source]
    subgraph_nodes = {source}
    for _ in range(depth):
        next_frontier = []
        for node in frontier:
            if not kg.has_node(node): continue
            deg = kg.degree(node)
            for _, nbr, data in kg.edges(node, data=True):
                rel = data.get("relation", "unknown")
                weight = RELATION_WEIGHTS.get(rel, 0)
                if (deg > HUB_DEGREE_THRESHOLD and weight >= 0.7) or (deg <= 5) or (5 < deg <= HUB_DEGREE_THRESHOLD and weight >= 0.3):
                    if nbr not in visited:
                        visited.add(nbr)
                        subgraph_nodes.add(nbr)
                        next_frontier.append(nbr)
        frontier = next_frontier
    return kg.subgraph(subgraph_nodes)

class HybridRetriever:
    def __init__(self, kg: nx.DiGraph, embeddings_path: str):
        self.kg = kg
        data = np.load(embeddings_path, allow_pickle=True)
        keys = list(data.keys())
        id_key = 'node_ids' if 'node_ids' in keys else keys[0]
        vec_key = keys[1] if keys[0] == id_key else keys[0]
        
        self.node_ids = data[id_key]
        vecs = data[vec_key].astype("float32")
        self.node_to_idx = {str(node): i for i, node in enumerate(self.node_ids)}
        faiss.normalize_L2(vecs)
        
        try:
            res = faiss.StandardGpuResources()
            self.index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(vecs.shape[1]))
        except Exception:
            self.index = faiss.IndexFlatIP(vecs.shape[1])
            
        self.index.add(vecs)
        self.embeddings = vecs

    def candidates(self, query_node: str, top_m: int = 15):
        if query_node not in self.node_to_idx: return []
        idx = self.node_to_idx[query_node]
        q = self.embeddings[idx].reshape(1, -1).copy()
        scores, indices = self.index.search(q, top_m)
        return [(str(self.node_ids[i]), float(s)) for i, s in zip(indices[0], scores[0])]

    def retrieve_evidence(self, source_node: str, depth: int = 2):
        symbolic_sub = type_aware_bfs(self.kg, source_node, depth)
        hard_nodes = set(symbolic_sub.nodes)
        hard_evidence = []
        for cand_node, sim in self.candidates(source_node, top_m=20):
            if cand_node in hard_nodes and cand_node != source_node:
                try:
                    path = nx.shortest_path(symbolic_sub, source_node, cand_node)
                    hard_evidence.append({"node": cand_node, "path": path, "similarity": sim})
                except nx.NetworkXNoPath:
                    pass
        return hard_evidence

print("2/4 Loading MedEmbed FAISS Index...")
retriever = HybridRetriever(prime_kg, os.path.join(DATA_DIR, "node_embeddings.npz"))

# B. Cross-Encoder
print("3/4 Loading MedCPT CrossEncoder...")
ranker = CrossEncoder("ncbi/MedCPT-Cross-Encoder", device="cuda" if torch.cuda.is_available() else "cpu")

def cascade_filter_paths(patient_context: str, candidate_paths: list, top_n: int = 5):
    if not candidate_paths: return []
    pairs = [(patient_context, " -> ".join(p["path"])) for p in candidate_paths if "path" in p]
    if not pairs: return candidate_paths[:top_n]
    scores = ranker.predict(pairs)
    ranked = sorted(zip(candidate_paths, scores), key=lambda x: x[1], reverse=True)
    return [p for p, _ in ranked[:top_n]]

# C. ChromaDB & Text Embedder
print("4/4 Initializing Oxford ChromaDB...")
if os.path.exists(CHROMA_WORK):
    shutil.rmtree(CHROMA_WORK)
shutil.copytree(CHROMA_SRC, CHROMA_WORK)

chroma_client = chromadb.PersistentClient(path=CHROMA_WORK)
chroma_collection = chroma_client.get_collection(name="oxford_oncology")
embedder = SentenceTransformer("abhinand/MedEmbed-large-v0.1", device="cuda" if torch.cuda.is_available() else "cpu")

print("All persistent resources successfully loaded!\n")


# ==========================================
# 3. METRIC SCORING & LLM-AS-A-JUDGE
# ==========================================
def calculate_recall(generated_text: str, ground_truth_items: list[str]) -> float:
    if not ground_truth_items: return 1.0
    text_lower = generated_text.lower()
    matches = sum(1 for item in ground_truth_items if item.lower() in text_lower)
    return matches / len(ground_truth_items)

def calculate_kg_citation_accuracy(top_paths: list[dict], generated_text: str) -> float:
    if not top_paths: return 1.0
    text_lower = generated_text.lower()
    cited = sum(1 for p in top_paths if str(p.get("node", "")).lower() in text_lower)
    return cited / len(top_paths)

def evaluate_with_llm_judge(patient_conv: str, ground_truth: dict, generated_pathway: str) -> dict:
    judge_prompt = f"""You are an expert Oncology Board Reviewer evaluating a Cancer RAG system's clinical output against reference Ground Truth standards.

PATIENT CASE:
{patient_conv}

REFERENCE GROUND TRUTH:
- Indicated Therapies: {ground_truth.get('indicated_therapies')}
- Mandatory Imaging: {ground_truth.get('mandatory_imaging')}
- Mandatory Workup/Labs: {ground_truth.get('mandatory_labs_workup')}

GENERATED CANCER RAG PATHWAY:
{generated_pathway}

Evaluate the output on three criteria from 1 (poor/unsafe) to 5 (flawless):
1. clinical_safety: Does the pathway avoid contraindicated therapies and dangerous errors?
2. staging_workup_accuracy: Did it order the correct digital imaging and laboratory tests?
3. kg_grounding_adherence: Are the recommended drugs and biomarkers appropriately aligned?

Return ONLY a valid JSON object with the following schema:
{{
  "clinical_safety": <1-5>,
  "staging_accuracy": <1-5>,
  "kg_grounding": <1-5>,
  "brief_justification": "<max 2 sentences>"
}}"""

    cfg = genai.GenerationConfig(response_mime_type="application/json", temperature=0.0)
    raw_eval = call_gemini_with_retry(judge_prompt, generation_config=cfg)
    try:
        return json.loads(raw_eval)
    except Exception:
        return {"clinical_safety": 3, "staging_accuracy": 3, "kg_grounding": 3, "brief_justification": "Evaluation parsed with fallback."}


# ==========================================
# 4. BENCHMARK EXECUTION LOOP
# ==========================================
# Load benchmark cases
with open(BENCHMARK_PATH, "r") as f:
    bench_cases = [json.loads(line) for line in f if line.strip()]

print(f"Starting benchmark evaluation across {len(bench_cases)} cases...\n")

benchmark_results = []

for idx, case in enumerate(bench_cases, 1):
    case_id = case.get("case_id", f"CASE_{idx}")
    conv = case["conversation"]
    gt = case["ground_truth"]
    
    print(f"\n================================================================================")
    print(f" EVALUATING [{idx}/{len(bench_cases)}]: {case_id} ({case.get('cancer_type', '').upper()})")
    print(f"================================================================================")
    
    # -----------------------------
    # Step A: PrimeKG Entity Routing
    # -----------------------------
    entities = extract_oncology_entities(conv)
    
    search_nodes = []
    if entities.cancer_type: search_nodes.append(entities.cancer_type)
    for bm in entities.biomarkers: search_nodes.append(bm.gene_or_marker)
    
    raw_hard_evidence = []
    for node_name in search_nodes:
        for t in [node_name, node_name.lower(), node_name.capitalize(), node_name.upper()]:
            if prime_kg.has_node(t):
                ev = retriever.retrieve_evidence(t, depth=2)
                raw_hard_evidence.extend(ev)
                break
                
    top_paths = cascade_filter_paths(conv, raw_hard_evidence, top_n=5)
    
    # -----------------------------
    # Step B: Intermediate DDX
    # -----------------------------
    ddx_prompt = f"""Given this oncology patient context, propose:
1. best_case: most indolent/localized plausible diagnosis and stage
2. worst_case: most aggressive/metastatic plausible diagnosis and stage
For each, list the staging workup, biomarker tests, and imaging needed. Ground in KG paths.

Patient context: {conv}
Hard evidence (verified KG paths): {json.dumps(top_paths, indent=2)}"""
    
    raw_ddx = call_gemini_with_retry(ddx_prompt)
    
    # -----------------------------
    # Step C: ChromaDB Retrieval & Synthesis
    # -----------------------------
    q_vec = embedder.encode([conv], convert_to_numpy=True, normalize_embeddings=True).tolist()
    chroma_res = chroma_collection.query(query_embeddings=q_vec, n_results=2)
    retrieved_docs = chroma_res["documents"][0]
    
    final_prompt = f"""You are a Chief Oncology Physician. 

PATIENT CONTEXT: 
{conv}

BRANCH A (Knowledge Graph Top Paths):
{top_paths}

BRANCH A (Initial DDX Output):
{raw_ddx}

BRANCH B (Oxford Handbook Evidence):
Context 1: {retrieved_docs[0] if len(retrieved_docs) > 0 else "N/A"}
Context 2: {retrieved_docs[1] if len(retrieved_docs) > 1 else "N/A"}

TASK:
Based on the above context, output a clear, structured Final Treatment Pathway:
1. Mandate the required digital imaging (e.g. MRI/PET).
2. Mandate the required lab work and blood tests.
3. Formulate the probable systemic therapy pathway, explicitly grounded in the Knowledge Graph paths.
"""
    
    final_pathway = call_gemini_with_retry(final_prompt)
    
    # -----------------------------
    # Step D: Scoring & LLM-as-a-Judge
    # -----------------------------
    workup_recall = calculate_recall(final_pathway, gt.get("mandatory_imaging", []) + gt.get("mandatory_labs_workup", []))
    therapy_recall = calculate_recall(final_pathway, gt.get("indicated_therapies", []))
    kg_citation_acc = calculate_kg_citation_accuracy(top_paths, final_pathway)
    
    judge_scores = evaluate_with_llm_judge(conv, gt, final_pathway)
    
    # Store metrics
    case_summary = {
        "case_id": case_id,
        "cancer_type": case.get("cancer_type"),
        "workup_recall": workup_recall,
        "therapy_recall": therapy_recall,
        "kg_citation_acc": kg_citation_acc,
        "judge_safety": judge_scores.get("clinical_safety", 0),
        "judge_staging": judge_scores.get("staging_accuracy", 0),
        "judge_grounding": judge_scores.get("kg_grounding", 0)
    }
    benchmark_results.append(case_summary)
    
    # Print Side-by-Side Outputs
    print("\n--- [GROUND TRUTH] ---")
    print(f"Primary Biomarker: {gt.get('primary_biomarker')}")
    print(f"Indicated Therapies: {gt.get('indicated_therapies')}")
    print(f"Mandatory Imaging: {gt.get('mandatory_imaging')}")
    print(f"Mandatory Workup: {gt.get('mandatory_labs_workup')}")
    
    print("\n--- [CANCER RAG OUTPUT] ---")
    print(final_pathway)
    
    print("\n--- [CASE EVALUATION & JUDGE SCORES] ---")
    print(f"• Workup Recall: {workup_recall*100:.1f}%")
    print(f"• Therapy Recall: {therapy_recall*100:.1f}%")
    print(f"• KG Citation Accuracy: {kg_citation_acc*100:.1f}%")
    print(f"• LLM-as-Judge Safety Score: {judge_scores.get('clinical_safety')}/5")
    print(f"• LLM-as-Judge Staging Score: {judge_scores.get('staging_accuracy')}/5")
    print(f"• LLM-as-Judge Grounding Score: {judge_scores.get('kg_grounding')}/5")
    print(f"• Judge Critique: {judge_scores.get('brief_justification')}")
    
    # Cleanup loop-level memory
    del raw_hard_evidence, top_paths, raw_ddx, final_pathway
    gc.collect()


# ==========================================
# 5. AGGREGATE BENCHMARK REPORT
# ==========================================
print("\n" + "="*80)
print("                       FINAL BENCHMARK SCORECARD")
print("="*80)

avg_workup = np.mean([r["workup_recall"] for r in benchmark_results]) * 100
avg_therapy = np.mean([r["therapy_recall"] for r in benchmark_results]) * 100
avg_citation = np.mean([r["kg_citation_acc"] for r in benchmark_results]) * 100
avg_safety = np.mean([r["judge_safety"] for r in benchmark_results])
avg_staging = np.mean([r["judge_staging"] for r in benchmark_results])
avg_grounding = np.mean([r["judge_grounding"] for r in benchmark_results])

print(f"{'Case ID':<10} | {'Cancer Type':<18} | {'Workup':<8} | {'Therapy':<8} | {'KG Cite':<8} | {'Safety':<6} | {'Staging':<7} | {'Grounding':<9}")
print("-" * 90)
for r in benchmark_results:
    print(f"{r['case_id']:<10} | {r['cancer_type']:<18} | {r['workup_recall']*100:>6.1f}% | {r['therapy_recall']*100:>6.1f}% | {r['kg_citation_acc']*100:>6.1f}% | {r['judge_safety']:>5.1f}  | {r['judge_staging']:>6.1f}  | {r['judge_grounding']:>8.1f}")
print("-" * 90)
print(f"{'OVERALL AVERAGE':<31} | {avg_workup:>6.1f}% | {avg_therapy:>6.1f}% | {avg_citation:>6.1f}% | {avg_safety:>5.2f}  | {avg_staging:>6.2f}  | {avg_grounding:>8.2f}")
print("="*80)

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


=== Initializing Persistent Graph & Vector Resources ===
1/4 Loading PrimeKG NetworkX Graph...
2/4 Loading MedEmbed FAISS Index...


3/4 Loading MedCPT CrossEncoder...


config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

All persistent resources successfully loaded!

Starting benchmark evaluation across 10 cases...


 EVALUATING [1/10]: ONC_001 (LUNG CANCER)

--- [GROUND TRUTH] ---
Primary Biomarker: EGFR
Indicated Therapies: ['Osimertinib', 'Erlotinib', 'Gefitinib']
Mandatory Imaging: ['Brain MRI', 'PET/CT', 'Chest CT']
Mandatory Workup: ['EBUS-TBNA', 'Pulmonary Function Tests', 'CMP', 'CBC']

--- [CANCER RAG OUTPUT] ---
As Chief Oncology Physician, I have synthesized the clinical context, Knowledge Graph (KG) insights, and evidentiary guidelines to establish the following treatment pathway for this 62-year-old patient.

### 1. Mandatory Digital Imaging & Staging
Given the high propensity for asymptomatic metastatic disease in lung adenocarcinoma, we must definitively stage the patient before determining if surgical resection is curative or if systemic palliative therapy is required.

*   **PET/CT (Whole Body):** Essential to assess the primary mass FDG-avidity and screen for occult mediastinal nodal 